# Titanic Survival Prediction — Logistic Regression from Scratch

**Target:** `Survived` (0 = did not survive, 1 = survived)

## 1. Imports and load the dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_curve, roc_auc_score, log_loss
)

df = pd.read_csv("titanic.csv")

print("Dataset shape:", df.shape)
display(df.head())
print("\\nData types and non-null counts:")
df.info()


## 2. Data quality and target distribution

In [ ]:
print("Missing values (largest first):")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["Survived"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], rot=0
)
axes[0].set_xticks([0, 1], ["Did not survive (0)", "Survived (1)"])
axes[0].set_ylabel("Number of passengers")
axes[0].set_title("Target class counts")

(df["Survived"].value_counts(normalize=True).sort_index() * 100).plot(
    kind="bar", ax=axes[1], rot=0
)
axes[1].set_xticks([0, 1], ["Did not survive (0)", "Survived (1)"])
axes[1].set_ylabel("Percentage (%)")
axes[1].set_title("Target class proportions")
plt.tight_layout()
plt.show()


## 3. Exploratory data visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

df["Age"].dropna().plot(kind="hist", bins=25, ax=axes[0, 0], edgecolor="black")
axes[0, 0].set_title("Passenger age distribution")
axes[0, 0].set_xlabel("Age")

df["Fare"].plot(kind="hist", bins=30, ax=axes[0, 1], edgecolor="black")
axes[0, 1].set_title("Ticket fare distribution")
axes[0, 1].set_xlabel("Fare")

survival_sex = df.groupby("Sex", observed=False)["Survived"].mean()
survival_sex.plot(kind="bar", ax=axes[1, 0], rot=0)
axes[1, 0].set_title("Survival rate by sex")
axes[1, 0].set_ylabel("Survival rate")
axes[1, 0].set_ylim(0, 1)

survival_class = df.groupby("Pclass")["Survived"].mean()
survival_class.plot(kind="bar", ax=axes[1, 1], rot=0)
axes[1, 1].set_title("Survival rate by passenger class")
axes[1, 1].set_xlabel("Passenger class")
axes[1, 1].set_ylabel("Survival rate")
axes[1, 1].set_ylim(0, 1)

plt.tight_layout()
plt.show()


## 4. Select features and split the data

We use a compact set of interpretable features: passenger class, sex, age, number of siblings/spouses, number of parents/children, fare, and port of embarkation. `PassengerId`, `Name`, `Ticket`, and `Cabin` are excluded from this first model; `Cabin` is also heavily missing.

Categorical variables are one-hot encoded. Numeric missing values are imputed with the **training-set median**; categorical missing values use the **training-set most frequent value**. Standardization is fitted only on the training data.

In [ ]:
target = df["Survived"].astype(int)

features = df[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)

numeric_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_cols = ["Sex", "Embarked"]

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train_num = pd.DataFrame(
    num_imputer.fit_transform(X_train_raw[numeric_cols]),
    columns=numeric_cols, index=X_train_raw.index
)
X_test_num = pd.DataFrame(
    num_imputer.transform(X_test_raw[numeric_cols]),
    columns=numeric_cols, index=X_test_raw.index
)

X_train_cat = pd.DataFrame(
    cat_imputer.fit_transform(X_train_raw[categorical_cols]),
    columns=categorical_cols, index=X_train_raw.index
)
X_test_cat = pd.DataFrame(
    cat_imputer.transform(X_test_raw[categorical_cols]),
    columns=categorical_cols, index=X_test_raw.index
)

X_train = pd.concat([X_train_num, X_train_cat], axis=1)
X_test = pd.concat([X_test_num, X_test_cat], axis=1)

X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True, dtype=float)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True, dtype=float)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

X_train = X_train.to_numpy(dtype=float)
X_test = X_test.to_numpy(dtype=float)
y_train = y_train.to_numpy(dtype=float)
y_test = y_test.to_numpy(dtype=int)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


## 5. Implement logistic regression with NumPy

The model computes

\[
z = Xw + b, \qquad \hat{y} = \sigma(z) = \frac{1}{1+e^{-z}}
\]

The binary cross-entropy loss is minimized using batch gradient descent. The gradients are

\[
\frac{\partial J}{\partial w} = \frac{1}{m}X^T(\hat{y}-y), \qquad
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m}(\hat{y}_i-y_i)
\]

Here, the output is a probability; we apply a threshold later to convert it into class predictions.

In [ ]:
def sigmoid(z):
    # Clip logits to avoid overflow in exp for extreme values.
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def binary_cross_entropy(y, probabilities):
    eps = 1e-12
    probabilities = np.clip(probabilities, eps, 1 - eps)
    return -np.mean(
        y * np.log(probabilities) + (1 - y) * np.log(1 - probabilities)
    )

m, n = X_train.shape
w = np.zeros(n, dtype=float)
b = 0.0

learning_rate = 0.05
iterations = 5000
loss_history = []

for i in range(iterations):
    probabilities = sigmoid(X_train @ w + b)
    loss = binary_cross_entropy(y_train, probabilities)
    loss_history.append(loss)

    error = probabilities - y_train
    dw = (X_train.T @ error) / m
    db = np.mean(error)

    w -= learning_rate * dw
    b -= learning_rate * db

print("Training complete")
print("Final training loss:", round(loss_history[-1], 5))
print("Bias:", round(b, 5))
print("Number of weights:", len(w))


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.xlabel("Iteration")
plt.ylabel("Binary cross-entropy loss")
plt.title("Gradient descent convergence")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Generate predictions on the test set

In [ ]:
y_test_prob = sigmoid(X_test @ w + b)
threshold = 0.5
y_test_pred = (y_test_prob >= threshold).astype(int)

print("First 10 actual labels:    ", y_test[:10])
print("First 10 predicted labels: ", y_test_pred[:10])
print("First 10 survival probs:   ", np.round(y_test_prob[:10], 3))


## 7. Classification metrics

Accuracy measures the overall fraction classified correctly. **Precision** measures how often predicted survivors actually survived; **recall** measures how many actual survivors were found; **F1** balances precision and recall. ROC-AUC measures ranking quality across thresholds. Because false negatives and false positives have different implications, do not judge the model using accuracy alone.

In [ ]:
metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 score", "ROC-AUC", "Log loss"],
    "Value": [
        accuracy_score(y_test, y_test_pred),
        precision_score(y_test, y_test_pred, zero_division=0),
        recall_score(y_test, y_test_pred, zero_division=0),
        f1_score(y_test, y_test_pred, zero_division=0),
        roc_auc_score(y_test, y_test_prob),
        log_loss(y_test, y_test_prob)
    ]
})
display(metrics.style.format({"Value": "{:.4f}"}))
print(classification_report(
    y_test, y_test_pred, target_names=["Did not survive", "Survived"], zero_division=0
))


In [ ]:
cm = confusion_matrix(y_test, y_test_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["Did not survive", "Survived"]
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Test-set confusion matrix")
plt.tight_layout()
plt.show()

fpr, tpr, thresholds = roc_curve(y_test, y_test_prob)
auc_value = roc_auc_score(y_test, y_test_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Scratch logistic regression (AUC = {auc_value:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random ranking")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate (recall)")
plt.title("ROC curve on test data")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Predicted probability distributions

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(y_test_prob[y_test == 0], bins=18, alpha=0.65, label="Actual: did not survive")
plt.hist(y_test_prob[y_test == 1], bins=18, alpha=0.65, label="Actual: survived")
plt.axvline(threshold, linestyle="--", label=f"Decision threshold = {threshold}")
plt.xlabel("Predicted probability of survival")
plt.ylabel("Number of passengers")
plt.title("Predicted probabilities by actual class")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Inspect learned coefficients

Coefficients are on standardized numeric features and one-hot encoded categories. A positive coefficient increases the log-odds of survival, holding the other features fixed; a negative coefficient decreases them. Coefficients describe this fitted model and should not automatically be interpreted as causal effects.

In [ ]:
# Recover the transformed feature names in the same order used for training.
feature_names = list(numeric_cols) + [
    col for col in X_train_raw.columns if col in categorical_cols
]
# One-hot columns are the categorical names after encoding.
feature_names = list(pd.get_dummies(
    pd.concat([
        pd.DataFrame(num_imputer.transform(X_train_raw[numeric_cols]), columns=numeric_cols, index=X_train_raw.index),
        pd.DataFrame(cat_imputer.transform(X_train_raw[categorical_cols]), columns=categorical_cols, index=X_train_raw.index)
    ], axis=1),
    columns=categorical_cols, drop_first=True, dtype=float
).columns)

coef_table = pd.DataFrame({"Feature": feature_names, "Coefficient": w})
coef_table = coef_table.sort_values("Coefficient")
display(coef_table)

plt.figure(figsize=(8, 5))
plt.barh(coef_table["Feature"], coef_table["Coefficient"])
plt.axvline(0, linewidth=1)
plt.xlabel("Coefficient")
plt.title("Learned logistic regression coefficients")
plt.tight_layout()
plt.show()


## 10. Interpretation and limitations

- The metrics are calculated on a held-out 20% test split; results can vary with the split.
- The 0.5 threshold is a default, not necessarily the best threshold for every objective.
- The dataset has missing values and historical biases; this is a learning exercise, not a causal analysis.
- This is a **from-scratch logistic regression** model. Scikit-learn is used only for data splitting, preprocessing utilities, and evaluation metrics—not to fit the classifier.
- Possible next experiments: compare thresholds, add passenger title/family-size features, use cross-validation, and compare with scikit-learn logistic regression as a benchmark.